In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_gold/00_dim_date
# Purpose  : Standard Materialized View (Batch) - independent of any
#            Delta streaming source, recomputed entirely on each pipeline run.
#            Execution cost is virtually zero due to small dataset size (~40k rows).
#            The Pipeline Advisor warning (QUERY_SOURCE_UNSUPPORTED) is expected
#            and acceptable in this context.
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.table(
    name="covid19_lakehouse.gold.dim_date",
    comment="Calendar dimension, one row per calendar date from 1909-01-01 to 2020-12-31",
)
def dim_date():
    return (
        spark.sql(
            """
            SELECT explode(sequence(
                to_date('1909-01-01'),
                to_date('2020-12-31'),
                interval 1 day
            )) AS full_date
            """
        )
        .select(
            F.date_format(F.col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
            F.col("full_date"),
            F.dayofmonth("full_date").alias("day"),
            F.date_format("full_date", "EEEE").alias("day_name"),
            F.month("full_date").alias("month"),
            F.date_format("full_date", "MMMM").alias("month_name"),
            F.quarter("full_date").alias("quarter"),
            F.year("full_date").alias("year"),
            F.weekofyear("full_date").alias("week_of_year"),
            F.dayofweek("full_date").isin([1, 7]).alias("is_weekend"),
        )
    )